In [ ]:
# Notebook setup
import sys
from pathlib import Path
base = Path.cwd()
source = next((p for p in (base, *base.parents) if (p / "code/pyproject.toml").is_file()), base / "Release-Client.zip")
if not source.exists() and "google.colab" in sys.modules:
    from google.colab import files
    print("Required file: Release-Client.zip (project code). Choose this ZIP; data files are requested next.", flush=True)
    files.upload(target_dir=str(base))
if not source.exists():
    raise FileNotFoundError("Supply Release-Client.zip, or open the extracted code/notebooks folder.")
sys.path.insert(0, str(source / "cost-aware-market-forecasting" if source.is_file() else source))
sys.modules.pop("run_zip", None)
from run_zip import prepare_reader
CODE = CODE_ROOT = prepare_reader('04_RQ1_E_indices_nine_model_benchmark.ipynb', source, ())

# 04 - RQ1_E: Index nine-model benchmark

## Methodology

Nine models use complete M15 bid/ask-derived index bars and three next-bar direction widths. Blocked 2024 out-of-fold predictions measure classification; H1-2025 selects width and confidence threshold. July 2025 to March 2026 provides descriptive forward economics using those fixed choices.

USA500 and USATECH are analysed separately. Net returns deduct fixed all-in round-trip costs of 2 bps for USA500 and 3 bps for USATECH, covering spread, commission and slippage; observed bar spreads are diagnostics, not a second deduction. Classification scores are medians across label widths. Q2-2026 results are reported separately in Notebook 22.

In [2]:
import json
import numpy as np
import pandas as pd
from IPython.display import display
from experiments.build_notebook_06 import MODEL_LABELS
from experiments.index_replication_protocol import MODEL_NAMES

STREAMS = ("usa500", "usatech")
ROOT = CODE_ROOT / "experiments" / "cache" / "index_replication"
FORWARD_ROOT = CODE_ROOT / "experiments" / "cache" / "index_all_model_forward"
classification, selected, forward = {}, {}, {}
for stream in STREAMS:
    result = json.loads((ROOT / stream / "result.json").read_text(encoding="utf-8"))
    diagnostic = json.loads((FORWARD_ROOT / stream / "result.json").read_text(encoding="utf-8"))
    assert result["q2_loaded"] is False
    assert diagnostic["q2_loaded"] is False and diagnostic["forward_rows"] == 36
    assert diagnostic["evidence_role"] == "secondary_reused_forward_diagnostic"
    classification[stream] = pd.read_parquet(ROOT / stream / "classification_2024.parquet")
    selected[stream] = pd.read_parquet(ROOT / stream / "h1_selected_policies.parquet")
    forward[stream] = pd.read_parquet(FORWARD_ROOT / stream / "forward_summary.parquet")
    assert classification[stream]["model_name"].nunique() == 9
    assert set(classification[stream]["model_name"]) == set(MODEL_NAMES)

## 2024 OOF model evidence

For each baseline model, the table reports the median classification score across the three registered label widths; economics are not selected on OOF F1.

In [3]:
rows = []
for stream in STREAMS:
    scope = classification[stream][classification[stream]["arm"].eq("selected_base")]
    view = scope.groupby("model_name", as_index=False).agg(
        macro_f1=("macro_f1", "median"),
        balanced_accuracy=("balanced_accuracy", "median"),
        short_recall=("short_recall", "median"),
        flat_recall=("flat_recall", "median"),
        long_recall=("long_recall", "median"),
    ).assign(Index=stream.upper())
    rows.append(view)
oof_view = pd.concat(rows, ignore_index=True).rename(columns={
    "model_name": "Model", "macro_f1": "Macro-F1", "balanced_accuracy": "Balanced accuracy",
    "short_recall": "SHORT recall", "flat_recall": "FLAT recall", "long_recall": "LONG recall",
})[["Index", "Model", "Macro-F1", "Balanced accuracy", "SHORT recall", "FLAT recall", "LONG recall"]]
oof_view["Model"] = oof_view["Model"].map(MODEL_LABELS)
oof_view = oof_view.sort_values(["Macro-F1", "Balanced accuracy"], ascending=[False, False])
display(oof_view.round(3))

,Index,Model,Macro-F1,Balanced accuracy,SHORT recall,FLAT recall,LONG recall
12,USATECH,LogReg,0.455,0.519,0.495,0.679,0.308
14,USATECH,MLP,0.439,0.503,0.431,0.661,0.358
9,USATECH,CatBoost,0.439,0.462,0.255,0.752,0.379
17,USATECH,XGBoost,0.438,0.483,0.298,0.677,0.469
8,USA500,XGBoost,0.437,0.486,0.256,0.778,0.405
0,USA500,CatBoost,0.436,0.463,0.195,0.850,0.344
13,USATECH,LSTM,0.435,0.495,0.458,0.645,0.349
3,USA500,LogReg,0.434,0.534,0.473,0.699,0.418
6,USA500,Random Forest,0.434,0.516,0.287,0.729,0.483
11,USATECH,GRU,0.433,0.501,0.602,0.677,0.294


## H1 policy calibration

January-June 2025 calibrates one label width and confidence threshold per model using trade count, side balance, positive months, Net, Sharpe and Sortino.

In [4]:
h1_rows = []
for stream in STREAMS:
    scope = selected[stream][selected[stream]["arm"].eq("selected_base")].copy()
    assert scope["model_name"].nunique() == 9 and set(scope["model_name"]) == set(MODEL_NAMES)
    scope["Index"] = stream.upper()
    h1_rows.append(scope)
h1_view = pd.concat(h1_rows, ignore_index=True).rename(columns={
    "model_name": "Model", "width_bps": "Width bps", "tau": "Threshold", "eligible": "H1 status",
    "trades": "Trades", "n_long": "LONG", "n_short": "SHORT", "positive_months": "Positive months",
    "net_return": "Net %", "daily_sharpe": "Sharpe", "daily_sortino": "Sortino",
})
h1_view["Net %"] *= 100
h1_view["Model"] = h1_view["Model"].map(MODEL_LABELS)
h1_view["H1 status"] = h1_view["H1 status"].map({True: "Pass", False: "Below gate"})
h1_view = h1_view.sort_values(["Net %", "Sortino", "Sharpe"], ascending=False)
display(h1_view[["Index", "Model", "Width bps", "Threshold", "H1 status", "Trades", "LONG", "SHORT", "Positive months", "Net %", "Sharpe", "Sortino"]].round(3))
passed = int(h1_view["H1 status"].eq("Pass").sum())

,Index,Model,Width bps,Threshold,H1 status,Trades,LONG,SHORT,Positive months,Net %,Sharpe,Sortino
9,USATECH,LogReg,10,0.65,Below gate,134,115,19,2,9.643,1.131,1.254
0,USA500,LogReg,15,0.80,Below gate,205,119,86,2,3.353,0.978,0.427
14,USATECH,CatBoost,15,0.75,Pass,110,85,25,4,2.986,0.738,0.325
6,USA500,MLP,15,0.60,Below gate,725,571,154,2,2.722,0.373,0.248
12,USATECH,Linear SVM,15,0.45,Below gate,443,413,30,1,1.503,0.118,0.063
17,USATECH,GRU,15,0.60,Below gate,172,158,14,3,0.981,0.224,0.108
15,USATECH,MLP,5,0.65,Below gate,152,36,116,3,0.804,0.116,0.032
4,USA500,XGBoost,5,0.75,Below gate,69,17,52,3,0.233,0.165,0.061
5,USA500,CatBoost,15,0.75,Below gate,256,200,56,3,-1.292,-0.233,-0.155
13,USATECH,XGBoost,10,0.75,Below gate,106,73,33,3,-2.878,-1.094,-0.408


## Forward results

Each model retains its H1-selected width and threshold for a common July 2025-March 2026 descriptive economic comparison.

In [5]:
active = pd.concat([
    forward[stream][forward[stream]["arm"].eq("selected_base")].assign(Index=stream.upper())
    for stream in STREAMS
], ignore_index=True).rename(columns={
    "model_name": "Model", "trades": "Trades", "n_long": "LONG", "n_short": "SHORT",
    "trades_per_day": "Trades/day", "net_return": "Net %", "daily_sharpe": "Sharpe",
    "daily_sortino": "Sortino", "max_drawdown": "Max drawdown %",
})
active["Model"] = active["Model"].map(MODEL_LABELS)
active["H1 status"] = active["h1_eligible"].map({True: "Pass", False: "Below gate"})
active["Forward status"] = active["status"].map({"traded": "Traded", "no_trades": "No trades"})
active["Net %"] *= 100; active["Max drawdown %"] *= 100
result_columns = ["Trades", "Trades/day", "LONG", "SHORT", "Net %", "Sharpe", "Sortino", "Max drawdown %"]
assert len(active) == 18 and active.groupby("Index")["Model"].nunique().eq(9).all()
assert np.isfinite(active[result_columns].to_numpy(dtype=float)).all()
active = active.sort_values(["Net %", "Sortino", "Model"], ascending=[False, False, True])
display(active[["Index", "Model", "H1 status", "Forward status", *result_columns]].round(3))
best = active.iloc[0]

,Index,Model,H1 status,Forward status,Trades,Trades/day,LONG,SHORT,Net %,Sharpe,Sortino,Max drawdown %
1,USA500,Decision Tree,Below gate,Traded,234,0.854,234,0,3.107,0.765,0.666,-3.734
6,USA500,MLP,Below gate,Traded,12,0.044,8,4,0.639,0.855,0.177,-0.456
10,USATECH,Decision Tree,Below gate,No trades,0,0.000,0,0,0.000,0.000,0.000,0.000
17,USATECH,GRU,Below gate,No trades,0,0.000,0,0,0.000,0.000,0.000,0.000
9,USATECH,LogReg,Below gate,No trades,0,0.000,0,0,0.000,0.000,0.000,0.000
4,USA500,XGBoost,Below gate,No trades,0,0.000,0,0,0.000,0.000,0.000,0.000
12,USATECH,Linear SVM,Below gate,Traded,1,0.004,0,1,-0.064,-1.154,-0.070,-0.064
5,USA500,CatBoost,Below gate,Traded,16,0.058,15,1,-0.195,-0.294,-0.059,-0.773
2,USA500,Random Forest,Below gate,Traded,18,0.066,17,1,-0.527,-0.709,-0.151,-0.965
8,USA500,GRU,Below gate,Traded,120,0.438,109,11,-0.815,-0.409,-0.146,-2.207


## Results

Only USATECH CatBoost meets the H1 adequacy rule; its forward net return is -1.007% over 19 trades. USA500 Decision Tree has the highest descriptive forward net return (+3.107%), but its 234 trades are all LONG and it failed H1 adequacy. Classification ranking therefore does not establish a stable economic advantage.

Takeaway: The strongest forward return is a diagnostic result, not evidence of an adequately supported trading strategy.